In [ ]:
# Lecture 17 — Spark SQL: temp views, spark.sql, and identical plans
# Spark 101 · Module 5 · Lecture 17
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- The same two DataFrames as the last lecture --------------------------
orders = spark.createDataFrame(
    [
        (1, 101, "Madrid",   19.99),
        (2, 101, "Madrid",   250.00),
        (3, 102, "Helsinki", 5.50),
        (4, 103, "New York", 1200.00),
        (5, 102, "Helsinki", 42.00),
    ],
    ["order_id", "customer_id", "city", "price"],
)
customers = spark.createDataFrame(
    [(101, "Ada"), (102, "Linus"), (103, "Grace")],
    ["customer_id", "name"],
)

In [ ]:
# --- Step 1: register temp views — a NAME for each DataFrame, not a copy ---
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")
# createOrReplace = safe to re-run this cell without an "already exists" error.

In [ ]:
# --- Step 2: query with plain SQL. The result is a DataFrame. --------------
result = spark.sql("""
    SELECT city, sum(price) AS total
    FROM orders
    GROUP BY city
    ORDER BY total DESC
""")
result.show()

In [ ]:
# The result is an ordinary DataFrame — chain API calls straight onto it.
result.filter("total > 100").show()

In [ ]:
# --- SQL joins read exactly like textbook SQL -----------------------------
spark.sql("""
    SELECT c.name, sum(o.price) AS spent
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.name
    ORDER BY spent DESC
""").show()

In [ ]:
# --- The key idea: same query, two doors, ONE plan ------------------------
from pyspark.sql.functions import sum as _sum

api_version = orders.groupBy("city").agg(_sum("price").alias("total"))
sql_version = spark.sql("SELECT city, sum(price) AS total FROM orders GROUP BY city")

print("=== DataFrame API plan ===")
api_version.explain()

print("=== Spark SQL plan ===")
sql_version.explain()
# Compare the two trees — same Exchange, same HashAggregate, same scan.
# Catalyst does not care which door you came through.

In [ ]:
# --- Mix freely: SQL -> DataFrame -> register again -> SQL ----------------
big = spark.sql("SELECT * FROM orders WHERE price > 10")   # SQL
big = big.withColumnRenamed("price", "amount")             # DataFrame API
big.createOrReplaceTempView("big_orders")                  # register the result
spark.sql("SELECT city, count(*) AS n FROM big_orders GROUP BY city").show()  # SQL again

In [ ]:
# A CTE (WITH) — build a named subquery, then query it.
spark.sql("""
    WITH city_rev AS (
        SELECT city, sum(price) AS total FROM orders GROUP BY city
    )
    SELECT * FROM city_rev WHERE total > 100 ORDER BY total DESC
""").show()

In [ ]:
# A window function: rank orders by price within each city.
spark.sql("""
    SELECT order_id, city, price,
           rank() OVER (PARTITION BY city ORDER BY price DESC) AS price_rank
    FROM orders
""").show()

In [ ]:
# CASE WHEN — bucket rows into labels.
spark.sql("""
    SELECT order_id, price,
           CASE WHEN price >= 100 THEN 'big' ELSE 'small' END AS size
    FROM orders
""").show()

In [ ]:
# Mix: SQL result -> DataFrame API view -> back to SQL.
agg = spark.sql("SELECT customer_id, sum(price) AS spent FROM orders GROUP BY customer_id")
agg.createOrReplaceTempView("spend")
spark.sql("SELECT c.name, s.spent FROM spend s JOIN customers c USING (customer_id) ORDER BY s.spent DESC").show()

In [ ]:
# Recap:
#   - createOrReplaceTempView names a DataFrame; spark.sql queries it.
#   - SQL returns a DataFrame — mix SQL and the API freely.
#   - Both doors compile to the SAME Catalyst plan — pick for readability.
# Next lecture: UDFs — and why they break this optimization.